# System ID using Gaussian Processes
by TimJanssen66

### General understanding
I'm using NARX, which is a method for formatting data in dynamic modelling (this case).
NARX model using GPs works through using GP to approximate or estimate a function that maps past outputs and past exogenous inputs to a current output.
GP-NARX learns from data creating both a predicted mean and measure of prediction uncertainty. 
1. Regressor (current output depends on non-linear mapping + noise)
2. Gaussian process prior is placed over the non-linear mapping --> what does this mean???
3. Find/Optimize for hyperparameters by maximizing marginal likelihood
4. Via OSA predictions, actual past output data is used and Free-simulation, future predictions use previous GP output predictions (uncertain)

Why is NARX better for (one-step) prediction than for (multi-step) simulation? 
Depends highly on source of its inputs and cascading accumulation of errors. In prediction (OSA), only measured historical (current and past/lagged) data is used to predict the next time step. Whereas in (MSA) simulation, the predicted value(s) are fed back into the system and because of that 
errors made in every next prediction accumulate. What predictions are used for the next prediction are based on how far we look back (while the next output also indirectly depends on data that goes further back).

Import relevant libraries

In [2]:
import numpy as np
import GPy
import time


### Shape of system ID data set

In [2]:
# Load the compressed npz file
data = np.load('training-val-test-data.npz')

# Print the available variables (keys) inside the file
print("Keys in npz file:", list(data.keys()))

# Assuming the typical keys are 'u' (inputs) and 'y' (outputs)
u = data['u']
th = data['th']

print(f"Shape of inputs (u): {u.shape}")
print(f"Shape of outputs (th): {th.shape}")
print(f"Total number of samples: {len(th)}")


Keys in npz file: ['u', 'th']
Shape of inputs (u): (35000,)
Shape of outputs (th): (35000,)
Total number of samples: 35000


### Partitioning data into training, validation and test sets

In [4]:
# 1. Load data
data = np.load('disc-benchmark-files/training-val-test-data.npz')
u = data['u'].flatten() # Flattening to 1D arrays if they are shape (N, 1)
th = data['th'].flatten()

N = len(th)

# 2. Define split fractions
train_frac = 0.70
val_frac = 0.15

# 3. Calculate chronological index boundaries
train_end = int(N * train_frac)
val_end = int(N * (train_frac + val_frac))

# 4. Slice the arrays into continuous chronological blocks
u_train, th_train = u[:train_end], th[:train_end]
u_val, th_val     = u[train_end:val_end], th[train_end:val_end]
u_test, th_test   = u[val_end:], th[val_end:]

# 5. Verify the block shapes
print(f"Train samples: {len(th_train)} ({train_frac*100:.0f}%)")
print(f"Val samples:   {len(th_val)} ({val_frac*100:.0f}%)")
print(f"Test samples:  {len(th_test)} ({(1 - train_frac - val_frac)*100:.0f}%)")


FileNotFoundError: [Errno 2] No such file or directory: 'disc-benchmark-files/training-val-test-data.npz'

# NARX-Sparse-GP
Questions:
1. Is the way I split the data correct?
2. Does the kernel depend on the structure of the measured data?
3. Should I try a sweep with the parameters?

In [7]:
# 1. LOAD AND PARTITION DATA
data = np.load('training-val-test-data.npz')
u_full = data['u'].flatten()
th_full = data['th'].flatten()
N_total = len(u_full)
# u and th have shape (35000,), essentially standard arrays

# Splitting data into 70% Training, 15% Validation, 15% Test sets
train_end = int(N_total * 0.70)
val_end = int(N_total * 0.85)

u_train, th_train = u_full[:train_end], th_full[:train_end]
u_val, th_val     = u_full[train_end:val_end], th_full[train_end:val_end]
u_test, th_test   = u_full[val_end:], th_full[val_end:]

# 2. NARX DATA FORMATTING
def create_IO_data(u, y, na, nb):
    # Creates the NARX feature matrix X and target vector Y
    # GPy requires 2D arrays, so Y is reshaped to (N, 1)
    X, Y = [], []
    for k in range(max(na, nb), len(y)):
        # Feature vector: [u[k-nb], ..., u[k-1], y[k-na], ..., y[k-1]]
        X.append(np.concatenate([u[k-nb:k], y[k-na:k]]))
        Y.append(y[k])
    
    return np.array(X), np.array(Y).reshape(-1, 1)

# Define lag orders (I think these can be tuned later using the validation set)
# previously had 2, 2
na = 2
nb = 2

X_train, Y_train = create_IO_data(u_train, th_train, na, nb)
X_val, Y_val     = create_IO_data(u_val, th_val, na, nb)

print(f"X_train shape: {X_train.shape}, Y_train shape: {Y_train.shape}")

# 3. SPARSE GAUSSIAN PROCESS SETUP
print("Initializing Sparse GP Model...")

# Initialize inducing points (Z)
# We randomly sample 'num_inducing' rows from X_train to act as our sparse support points
# first I had 150
num_inducing = 500 
random_indices = np.random.choice(X_train.shape[0], num_inducing, replace=False)
Z = X_train[random_indices, :].copy()

# Define RBF kernel with ARD (Automatic Relevance Determination) 
# allows a different lengthscale for each NARX feature
kernel = GPy.kern.RBF(input_dim=X_train.shape[1], ARD=True)
# Other Kernels to try?!:
# Period kernel
# Matérn Kernel (Specifically Matérn 3/2 or 5/2)
# Rotational quadratic
# Composite (Physics-Informed)
# Neural Network (ArcCosine)

# Construct the Sparse GP Regressor
m_sparse = GPy.models.SparseGPRegression(X_train, Y_train, kernel=kernel, Z=Z)

# Optimize hyperparameters (lengthscales, variance and inducing point locations)
print("Optimizing model (time depending on num_inducing)...")
start_time = time.time()
m_sparse.optimize('bfgs', messages=False) 
# Print the optimized hyperparameter values
print(m_sparse) 
print(f"Optimization finished in {time.time() - start_time:.2f} seconds.")

# 4. OSA PREDICTION (VALIDATION)
# GPy predict() returns a tuple: (Mean, Variance). We only need the Mean [0] for error calculation
Y_val_pred, Y_val_var = m_sparse.predict(X_val)

print('\nValidation 1 step ahead prediction errors:')
rmse_val = np.mean((Y_val_pred - Y_val)**2)**0.5
rmse_val_deg = rmse_val*(180/np.pi)
nrms_val = rmse_val / np.std(Y_val)
# print(f'RMS: {rmse_val:.4f} radians')
# print(f'RMS: {rmse_val_deg:.4f} degrees')
print(f'NRMS: {nrms_val * 100:.2f}%')

# 5. SIMULATION LOOP (INFINITE HORIZON)
def simulation_IO_model(model_predict_func, ulist, ylist_init, na, nb, skip=50):
    # Simulates the system by feeding its own past predictions back into the model 
    # Initialize the past inputs and outputs using true data up to the 'skip' index
    upast = ulist[skip-nb:skip].tolist()
    ypast = ylist_init[skip-na:skip].tolist()
    
    Y_simulated = ylist_init[:skip].tolist()
    
    for u in ulist[skip:]:
        # Create the current feature vector
        x = np.concatenate([upast, ypast], axis=0)
        
        # Predict the next output 
        ypred = model_predict_func(x)
        Y_simulated.append(ypred)
        
        # Shift the delay buffers 
        upast.append(u)
        upast.pop(0)
        ypast.append(ypred)
        ypast.pop(0)
        
    return np.array(Y_simulated)

# Wrap the GPy prediction method for the simulation loop
# It forces the 1D input 'x' into a 2D array of shape (1, D), calls predict and extracts the single scalar mean value [0][0, 0]
gp_predict_wrapper = lambda x: m_sparse.predict(np.array([x]))[0][0, 0]

print("\nRunning closed-loop simulation on validation data...")
skip_idx = max(na, nb)
th_val_sim = simulation_IO_model(gp_predict_wrapper, u_val, th_val, na, nb, skip=skip_idx)

print('Validation simulation errors:')
rmse_sim = np.mean((th_val_sim[skip_idx:] - th_val[skip_idx:])**2)**0.5
rmse_sim_deg = rmse_sim*(180/np.pi)
nrms_sim = rmse_sim / np.std(th_val)
# print(f'RMS:  {rmse_sim:.4f} radians')
# print(f'RMS:  {rmse_sim_deg:.4f} degrees')
print(f'NRMS: {nrms_sim * 100:.2f}%')

# 6. GENERATE RESULTS TABLES
# Simulation results wrt benchmark
sim_data = [
    ["Lower bound", 0.0195, 1.12],
    ["Good NN model", 0.0271, 1.55],
    ["Linear Model", 0.255, 14.6],
    ["Sparse GP Model", rmse_sim, rmse_sim_deg]
]

# Prediction results wrt benchmark
pred_data = [
    ["Good NN model", 0.00382, 0.219],
    ["Linear Model", 0.00665, 0.381],
    ["Sparse GP Model", rmse_val, rmse_val_deg]
]

# Sort the data by RMSE (radians) so the most accurate models are at the top
sim_data.sort(key=lambda x: x[1])
pred_data.sort(key=lambda x: x[1])

# Print tables
def print_table(title, data):
    print(f"{title}:")
    print("-" * 55)
    print(f"{'Type':<20} | {'RMSE (radians)':<15} | {'RMSE (deg)':<15}")
    print("-" * 55)
    for row in data:
        print(f"{row[0]:<20} | {row[1]:<15.4g} | {row[2]:<15.4g}")
    print("-" * 55 + "\n")

print("\n")
print_table("Simulation Results", sim_data)
print_table("Prediction Results", pred_data)

X_train shape: (24498, 4), Y_train shape: (24498, 1)
Initializing Sparse GP Model...
Optimizing model (time depending on num_inducing)...

Name : sparse_gp
Objective : -100977.74749159813
Number of Parameters : 2006
Number of Optimization Parameters : 2006
Updates : True
Parameters:
  sparse_gp.               |                   value  |  constraints  |  priors
  inducing_inputs          |                (500, 4)  |               |        
  rbf.variance             |      1.4079529799962174  |      +ve      |        
  rbf.lengthscale          |                    (4,)  |      +ve      |        
  Gaussian_noise.variance  |  1.4621411427067506e-05  |      +ve      |        
Optimization finished in 779.70 seconds.

Validation 1 step ahead prediction errors:
NRMS: 0.79%

Running closed-loop simulation on validation data...
Validation simulation errors:
NRMS: 10.90%


Simulation Results:
-------------------------------------------------------
Type                 | RMSE (radians)  | RMS